## Store Sales Forecasting Pipeline - Delhi Data Scientist Assignment
- **Author -** Shivam Chaudhary
- **Objective -** Data preprocessing, feature engineering, LSTM model training, and exporting artifacts for deployment.

In [18]:
# %% [markdown]
# ## High-Performance Responsive Store Sales Forecasting Pipeline
# **Author:** Shivam  
# **Objective:** Gated Recurrent Unit (GRU) with a highly responsive lookback window.

# %%
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
from sklearn.preprocessing import StandardScaler

# %%
# 1. Ingest Dataset and Standardize Formats
print("Starting dataset ingestion engine...")
df_raw = pd.read_excel('Store Sales Hoi Assignment Data Scientist.xlsx', sheet_name='Sheet1')
df_raw.columns = df_raw.columns.astype(str)

df_clean = df_raw.copy()
df_clean.iloc[:, 1:] = df_clean.iloc[:, 1:].ffill().bfill().fillna(0)

store_columns = [col for col in df_raw.columns if col != 'Date']
num_stores = len(store_columns)
df_clean[store_columns] = df_clean[store_columns].clip(lower=0)

# %%
# 2. Causal Feature Engineering
def build_features(df):
    data = df.copy()
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['IsWeekend'] = data['DayOfWeek'].isin([5, 6]).astype(int)
    data['Month'] = data['Date'].dt.month
    
    data['Sin_Month'] = np.sin(2 * np.pi * data['Month'] / 12)
    data['Cos_Month'] = np.cos(2 * np.pi * data['Month'] / 12)
    
    data['IsHoliday'] = data['Date'].apply(
        lambda x: 1 if (x.month == 1 and x.day == 26) or 
                       (x.month == 8 and x.day == 15) or 
                       (x.month == 10 and x.day == 2) or 
                       (x.month == 12 and x.day == 25) else 0
    )
    
    delhi_temp_profile = {1: 15, 2: 19, 3: 25, 4: 31, 5: 34, 6: 35, 7: 31, 8: 30, 9: 30, 10: 28, 11: 20, 12: 15}
    data['Delhi_Temp'] = data['Month'].map(delhi_temp_profile)
    return data

df_features = build_features(df_clean)
exo_columns = ['IsWeekend', 'Sin_Month', 'Cos_Month', 'IsHoliday', 'Delhi_Temp']

# %%
# 3. Isolated Training Data Preparation
train_df = df_features[df_features['Date'] < '2026-04-01'].reset_index(drop=True)

# Apply 99th Percentile Outlier Mitigation purely on Training Targets to resolve target mean distortion
for col in store_columns:
    limit = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(upper=limit)

sales_scaler = StandardScaler()
feature_scaler = StandardScaler()

sales_scaler.fit(train_df[store_columns])
feature_scaler.fit(train_df[exo_columns])

scaled_full_sales = sales_scaler.transform(df_features[store_columns])
scaled_full_exo = feature_scaler.transform(df_features[exo_columns])
scaled_master_matrix = np.hstack([scaled_full_sales, scaled_full_exo])

# %%
# 4. Generating High-Responsiveness Lookback Tensors
def generate_tensors(df_source, master_matrix, window_size, target_dim):
    x_seq, y_seq = [], []
    indices = df_source.index
    for idx in indices:
        if idx < window_size:
            continue
        x_seq.append(master_matrix[idx - window_size : idx, :])
        y_seq.append(master_matrix[idx, :target_dim])
    return torch.tensor(np.array(x_seq), dtype=torch.float32), torch.tensor(np.array(y_seq), dtype=torch.float32)

# Optimized lookback window size to respond dynamically to sudden baseline changes
WINDOW_SIZE = 3
X_train_t, y_train_t = generate_tensors(train_df, scaled_master_matrix, WINDOW_SIZE, num_stores)
X_val_t, y_val_t = generate_tensors(df_features[(df_features['Date'] >= '2026-04-01') & (df_features['Date'] <= '2026-04-12')], scaled_master_matrix, WINDOW_SIZE, num_stores)

# %%
# 5. Robust Gated Recurrent Unit (GRU) Model Definition
class RobustGRUForecaster(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RobustGRUForecaster, self).__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

model = RobustGRUForecaster(input_size=scaled_master_matrix.shape[1], hidden_size=16, output_size=num_stores)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# %%
# 6. Optimized Model Training Loop
MAX_EPOCHS = 35
print("Running optimized training iterations...")
for epoch in range(MAX_EPOCHS):
    model.train()
    optimizer.zero_grad()
    
    train_preds = model(X_train_t)
    loss = criterion(train_preds, y_train_t)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        val_preds = model(X_val_t)
        
    train_mae = torch.mean(torch.abs(train_preds - y_train_t)).item()
    val_mae = torch.mean(torch.abs(val_preds - y_val_t)).item()
    
    train_accuracy = max(0, 100 * (1 - (train_mae / (torch.mean(torch.abs(y_train_t)).item() + 1e-5))))
    val_accuracy = max(0, 100 * (1 - (val_mae / (torch.mean(torch.abs(y_val_t)).item() + 1e-5))))
    
    print(f"Epoch [{epoch+1:02d}/{MAX_EPOCHS}] | Loss: {loss.item():.5f} | Train Acc: {train_accuracy:.2f}% | Test/Val Acc: {val_accuracy:.2f}%")

# %%
# 7. Package and Save Runtime Artifacts
print("\nExporting pipeline model weights...")
torch.save(model.state_dict(), 'lstm_store_model.pth')

with open('sales_scaler.pkl', 'wb') as f: pickle.dump(sales_scaler, f)
with open('feature_scaler.pkl', 'wb') as f: pickle.dump(feature_scaler, f)

metadata = {'store_columns': store_columns, 'exo_columns': exo_columns, 'window_size': WINDOW_SIZE}
with open('metadata.pkl', 'wb') as f: pickle.dump(metadata, f)
print("All artifacts successfully saved!")

Starting dataset ingestion engine...
Running optimized training iterations...
Epoch [01/35] | Loss: 1.26160 | Train Acc: 0.00% | Test/Val Acc: 6.92%
Epoch [02/35] | Loss: 1.17889 | Train Acc: 0.00% | Test/Val Acc: 9.88%
Epoch [03/35] | Loss: 1.11091 | Train Acc: 3.74% | Test/Val Acc: 12.47%
Epoch [04/35] | Loss: 1.05011 | Train Acc: 7.22% | Test/Val Acc: 15.01%
Epoch [05/35] | Loss: 0.99294 | Train Acc: 10.67% | Test/Val Acc: 18.04%
Epoch [06/35] | Loss: 0.93730 | Train Acc: 14.35% | Test/Val Acc: 21.70%
Epoch [07/35] | Loss: 0.88328 | Train Acc: 18.32% | Test/Val Acc: 25.16%
Epoch [08/35] | Loss: 0.83246 | Train Acc: 22.42% | Test/Val Acc: 28.04%
Epoch [09/35] | Loss: 0.78669 | Train Acc: 26.37% | Test/Val Acc: 30.12%
Epoch [10/35] | Loss: 0.74726 | Train Acc: 30.04% | Test/Val Acc: 31.16%
Epoch [11/35] | Loss: 0.71470 | Train Acc: 33.35% | Test/Val Acc: 31.15%
Epoch [12/35] | Loss: 0.68867 | Train Acc: 36.10% | Test/Val Acc: 30.80%
Epoch [13/35] | Loss: 0.66820 | Train Acc: 38.00% | 